In [1]:
!pip install bokeh


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import HoverTool, Legend
from bokeh.io import output_notebook
from bokeh.layouts import column
import numpy as np

# Enable Bokeh to display in notebooks
output_notebook()

Loading BokehJS ...

In [3]:
df = pd.read_csv('/Users/rileybitterli/Downloads/MTA_Daily_Ridership_and_Traffic__Beginning_2020_20260126.csv')

In [4]:
df

,Date,Mode,Count
0,01/25/2026,AAR,"8,401"
1,01/25/2026,BT,"168,610"
2,01/25/2026,Bus,"195,423"
3,01/25/2026,LIRR,"30,577"
4,01/25/2026,MNR,"27,652"
...,...,...,...
15856,03/01/2020,SIR,"1,636"
15857,03/01/2020,LIRR,"86,790"
15858,03/01/2020,Bus,"984,908"
15859,03/01/2020,BT,"690,074"


In [5]:
# Clean up the dataframe
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Strip commas from Count column and convert to integer
# Handle NaN values by using nullable integer type (Int64)
if df['Count'].dtype == 'object':
    # Remove commas and convert to numeric (handles NaN)
    df['Count'] = pd.to_numeric(df['Count'].str.replace(',', '', regex=False), errors='coerce')
else:
    # If already numeric, ensure it handles NaN
    df['Count'] = pd.to_numeric(df['Count'], errors='coerce')

# Convert to nullable integer type (Int64) - preserves NaN values
df['Count'] = df['Count'].astype('Int64')

# Display info about cleaned dataframe
print(f"Dataframe shape: {df.shape}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nNull values in Count: {df['Count'].isna().sum()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()


Dataframe shape: (15861, 3)

Date range: 2020-03-01 00:00:00 to 2026-01-25 00:00:00

Null values in Count: 38

Data types:
Date     datetime64[ns]
Mode             object
Count             Int64
dtype: object

First few rows:


,Date,Mode,Count
0,2026-01-25,AAR,8401
1,2026-01-25,BT,168610
2,2026-01-25,Bus,195423
3,2026-01-25,LIRR,30577
4,2026-01-25,MNR,27652


In [6]:
df = df[df['Mode'] == 'Subway']

In [7]:
# Check data frequency - are we getting daily data?
print(f"Total rows after filtering to Subway: {len(df)}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nUnique dates: {df['Date'].nunique()}")
print(f"\nSample of dates to check frequency:")
print(df[['Date', 'Count']].head(20))
print(f"\nChecking for date patterns:")
date_diffs = df['Date'].diff().dt.days
print(f"Date differences (days): {date_diffs.value_counts().head(10)}")


Total rows after filtering to Subway: 2157

Date range: 2020-03-01 00:00:00 to 2026-01-25 00:00:00

Unique dates: 2157

Sample of dates to check frequency:
          Date    Count
6   2026-01-25   817201
8   2026-01-24  2215771
15  2026-01-23  3774773
22  2026-01-22  4120723
34  2026-01-21  4025125
38  2026-01-20  3877378
45  2026-01-19  2453191
54  2026-01-18  1892969
63  2026-01-17  2366104
72  2026-01-16  3777231
81  2026-01-15  4122435
90  2026-01-14  4123499
99  2026-01-13  4108805
108 2026-01-12  3770308
117 2026-01-11  1978534
126 2026-01-10  2395364
135 2026-01-09  3763800
144 2026-01-08  4096415
153 2026-01-07  4074540
162 2026-01-06  3973013

Checking for date patterns:
Date differences (days): Date
-1.0    2156
Name: count, dtype: int64


In [8]:
df.sort_values('Date', ascending=False).head(20)

,Date,Mode,Count
6,2026-01-25,Subway,817201
8,2026-01-24,Subway,2215771
15,2026-01-23,Subway,3774773
22,2026-01-22,Subway,4120723
34,2026-01-21,Subway,4025125
38,2026-01-20,Subway,3877378
45,2026-01-19,Subway,2453191
54,2026-01-18,Subway,1892969
63,2026-01-17,Subway,2366104
72,2026-01-16,Subway,3777231


In [15]:
import plotly.express as px


# Sort by date and calculate rolling average
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df['Rolling_7d_Avg'] = df['Count'].rolling(window=7).mean()

# Interactive plot
fig = px.line(
    df,
    x='Date',
    y='Rolling_7d_Avg',
    markers=True,
    title='Subway Ridership - 7 Day Rolling Average'
)

fig.update_traces(
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>7-Day Avg: %{y:,.0f}<extra></extra>'
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Ridership (7-Day Rolling Avg)',
    yaxis_tickformat=',.0f',
    hovermode='x unified'
    
)
fig.show(config={'modeBarButtonsToAdd': ['toggleFullScreen']})
fig.write_html('chart.html')



In [18]:
import webbrowser
import os

fig.write_html('chart.html')
webbrowser.open('file://' + os.path.abspath('chart.html'))

True

In [10]:
df['Year'] = df['Date'].dt.year
highest_by_year = df.loc[df.groupby('Year')['Rolling_7d_Avg'].idxmax(), ['Date', 'Rolling_7d_Avg']]
highest_by_year['Delta_From_Prev_Year'] = highest_by_year['Rolling_7d_Avg'].diff()
highest_by_year['Rolling_7d_Avg'] = highest_by_year['Rolling_7d_Avg'].round().astype(int)
highest_by_year['Delta_From_Prev_Year'] = highest_by_year['Delta_From_Prev_Year'].round().astype('Int64')

In [11]:
highest_by_year

,Date,Rolling_7d_Avg,Delta_From_Prev_Year
15813,2020-03-07,4574761,<NA>
11298,2021-12-12,2975653,-1599108
8757,2022-12-10,3338096,362442
6503,2023-10-28,3507260,169164
3612,2024-12-14,3821409,314149
738,2025-11-03,3997716,176307
72,2026-01-16,3468025,-529691
